# Kaggle – Data on the top — 04 · Modo competición 🏁 (blend)

Tu profe ha comprado una tienda de portátiles y quiere un modelo de ML para fijar precios.

**Este notebook (extra, 4/3 😄):** entrega adicional en *modo competición* — el objetivo es exprimir el RMSE
con todas las palancas legítimas, manteniendo la disciplina de validación de los notebooks anteriores:

1. **Feature engineering v2**: rescatamos `Product` (que los notebooks 1–3 descartaban) como *familia* de producto,
   y añadimos generación/sufijo de CPU, número de modelo de GPU y otros detalles.
2. **Bake-off multi-modelo**: XGBoost, LightGBM, CatBoost (y variantes en `log`) evaluados con OOF.
3. **Blend con pesos óptimos (NNLS)** ajustados sobre OOF de dos semillas de folds y **evaluados en semillas
   vírgenes** que no participan en ninguna decisión (el mismo antídoto anti-*winner's curse* del notebook 03).
4. **Promediado de semillas** dentro de cada modelo y **parche de clones exactos** como remate.

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from scipy.optimize import nnls

# Falsa alarma conocida de LightGBM dentro de Pipelines: en fit se inventa nombres
# sintéticos (Column_0...) y sklearn avisa en cada predict, aunque los datos de fit
# y predict pasan por el mismo preprocesador (son consistentes). Es cosmético y no
# afecta a las predicciones; lo silenciamos con un filtro específico de ese mensaje.
warnings.filterwarnings('ignore', message='X does not have valid feature names')

## 2. Datos

La exploración completa está en los notebooks 01–03 (sin nulos, precio sesgado, señal en las columnas de texto).
Aquí vamos directos al grano.

> **Sin holdout esta vez:** la validación es 100% *out-of-fold* (OOF) con varias semillas de folds. El papel del
> holdout lo hacen **semillas de folds vírgenes** (que no se usan para ninguna decisión) en la sección 4.4.

In [2]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')
y = df['Price_in_euros']
X_raw = df.drop(columns=['Price_in_euros'])
X_raw.shape, y.shape

((912, 12), (912,))

## 3. Procesado de datos — feature engineering v2

El v1 (notebooks 1–3) **traducía formato**: unidades atrapadas en texto, resolución → PPI, almacenamiento por tipo. El v2 va a por una señal distinta: la **identidad y la posición en la gama**. El precio no lo fijan solo las specs — lo fija *quién es* la máquina: su línea de producto, su generación, su posicionamiento (un ThinkPad y un IdeaPad con specs gemelas no valen lo mismo).

La estrella es `family`, y el matiz importante es la **granularidad**: `Product` crudo (480 valores ≈ identificadores casi únicos) sería *memorizar*; su primer token (~86 familias con decenas de ejemplos) es *generalizar* — transfiere a cualquier portátil nuevo de una familia conocida. Los notebooks 1–3 descartaban la columna entera; aquí rescatamos su parte generalizable.

| Feature nueva | Cómo | Por qué mueve el precio |
|---|---|---|
| `family` | 1er token de `Product` ("thinkpad", "rog", "xps"...) | Línea de producto = posicionamiento de mercado |
| `cpu_gen` | "i7 **7**700HQ" → 7 (solo Intel Core) | Generación ≈ antigüedad: a misma gama, más nuevo = más caro |
| `cpu_suffix` | 7700**HQ** → U/HQ/HK/M/T/Y | Clase de consumo: un i7-**U** puede rendir menos que un i5-**HQ**; "i7 + GHz" lo oculta |
| `gpu_num` | GTX **1050**, HD **620** → número | Posición en la gama dentro de cada marca; el árbol corta por umbrales |
| `gpu_gtx` / `gpu_quadro` / `gpu_dedicated` | flags; dedicada = marca ≠ Intel | Tiers sin ambigüedad (sustituyen la regex difusa del v1) |
| `retina` | flag en `ScreenResolution` | Panel premium de Apple |
| `OpSys` crudo | 9 categorías sin simplificar | "Windows 7" ≈ flota empresarial; "10 S" ≈ gama básica |
| `weight_per_inch` | peso / pulgadas | Densidad ≈ calidad de construcción |

El parseo sigue siendo **fila a fila** (sin `.fit`) → aplicable a `test.csv` sin *leakage*. `family` entra por tres vías según el miembro del blend — one-hot, `TargetEncoder` (media de precio suavizada con *cross-fitting* interno: cada fila recibe estadísticas calculadas **sin ella misma**) y categórica nativa de CatBoost — tres tratamientos de la misma señal que además aportan diversidad al ensemble.

**Resultado (adelanto):** con el mismo XGBoost, pasar de features v1 a v2 baja la CV de ~257 a ~227 € — **−30 € solo de features**, antes de tocar ningún modelo. El experimento es limpio: entre ambos solo cambian las columnas.

In [3]:
def cpu_category(s):
    """Gama de CPU a partir del texto."""
    s = s.lower()
    for k, v in [('i7','i7'), ('i5','i5'), ('i3','i3'), ('celeron','celeron'),
                 ('pentium','pentium'), ('atom','atom'), ('xeon','xeon'), ('ryzen','ryzen')]:
        if k in s: return v
    return 'amd_other' if 'amd' in s else 'other'

def parse_memory(mem):
    """'128GB SSD +  1TB HDD' -> GB por tipo de almacenamiento."""
    out = {'ssd': 0.0, 'hdd': 0.0, 'flash': 0.0, 'hybrid': 0.0}
    for parte in mem.split('+'):
        m = re.search(r'([\d\.]+)\s*(TB|GB)', parte)
        if not m:
            continue
        size = float(m.group(1)) * (1000 if m.group(2) == 'TB' else 1)
        if   'SSD'    in parte: out['ssd']    += size
        elif 'HDD'    in parte: out['hdd']    += size
        elif 'Hybrid' in parte: out['hybrid'] += size
        elif 'Flash'  in parte: out['flash']  += size
    return pd.Series(out)

def feature_engineering_v2(df):
    """FE v1 (notebooks 1-3) + familia de producto, detalle de CPU/GPU y extras.

    100% fila-a-fila: sin .fit ni estadisticas globales -> sin data leakage.
    """
    df = df.copy()
    df['ram'] = df['Ram'].str.replace('GB', '', regex=False).astype(int)
    df['weight'] = df['Weight'].str.replace('kg', '', regex=False).astype(float)
    # pantalla
    res = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)').astype(float)
    df['res_w'], df['res_h'] = res[0], res[1]
    df['ppi'] = np.sqrt(df['res_w']**2 + df['res_h']**2) / df['Inches']
    df['pixels'] = df['res_w'] * df['res_h']
    df['touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)
    df['ips'] = df['ScreenResolution'].str.contains('IPS').astype(int)
    df['retina'] = df['ScreenResolution'].str.contains('Retina').astype(int)
    # cpu
    df['cpu_ghz'] = df['Cpu'].str.extract(r'([\d\.]+)GHz').astype(float)
    df['cpu_cat'] = df['Cpu'].apply(cpu_category)
    df['cpu_brand'] = df['Cpu'].str.split().str[0]
    cm = df['Cpu'].str.extract(r'(\d{4})(U|HQ|HK|M|T|Y)?')
    is_core = df['Cpu'].str.contains('Core i', regex=False)
    df['cpu_gen'] = np.where(is_core, cm[0].str[0].astype(float), 0)
    df['cpu_gen'] = df['cpu_gen'].fillna(0)
    df['cpu_suffix'] = np.where(is_core, cm[1].fillna('none'), 'na')
    # gpu
    df['gpu_brand'] = df['Gpu'].str.split().str[0]
    df['gpu_num'] = df['Gpu'].str.extract(r'(\d{3,4})').astype(float).fillna(0)
    df['gpu_gtx'] = df['Gpu'].str.contains('GTX').astype(int)
    df['gpu_quadro'] = df['Gpu'].str.contains('Quadro').astype(int)
    df['gpu_dedicated'] = (df['gpu_brand'] != 'Intel').astype(int)
    # almacenamiento
    df = pd.concat([df, df['Memory'].apply(parse_memory)], axis=1)
    df['storage_total'] = df['ssd'] + df['hdd'] + df['flash'] + df['hybrid']
    df['has_ssd'] = (df['ssd'] > 0).astype(int)
    # producto y extras
    df['family'] = df['Product'].str.extract(r'^([A-Za-z]+|\d+)')[0].str.lower()
    df['weight_per_inch'] = df['weight'] / df['Inches']
    return df

X_fe = feature_engineering_v2(X_raw)

NUM = ['Inches', 'ram', 'weight', 'res_w', 'res_h', 'ppi', 'pixels', 'touchscreen', 'ips',
       'retina', 'cpu_ghz', 'cpu_gen', 'gpu_num', 'gpu_gtx', 'gpu_quadro', 'gpu_dedicated',
       'ssd', 'hdd', 'flash', 'hybrid', 'storage_total', 'has_ssd', 'weight_per_inch']
CAT_LOW = ['Company', 'TypeName', 'cpu_cat', 'cpu_brand', 'cpu_suffix', 'gpu_brand', 'OpSys']
CAT_TE  = ['family']
ALL = NUM + CAT_LOW + CAT_TE

X = X_fe[ALL]
print(f'{len(NUM)} numéricas + {len(CAT_LOW)} categóricas OHE + {len(CAT_TE)} target-encoded')
print(f'familias de producto: {X["family"].nunique()}')

23 numéricas + 7 categóricas OHE + 1 target-encoded
familias de producto: 83


**Codificación.** Las categóricas de baja cardinalidad van con One-Hot; `family` (~86 valores) va con
**`TargetEncoder`** de sklearn: sustituye cada familia por su precio medio *suavizado*, calculado con
**cross-fitting interno** (cada fila recibe la media de folds en los que ella no está) → sin *leakage*
aunque codifiquemos con el target. CatBoost no necesita nada de esto: come categorías nativas con
*ordered target statistics*, su especialidad.

In [4]:
def make_pre(te=True, scale=False):
    pasos = [('num', StandardScaler() if scale else 'passthrough', NUM),
             ('ohe', OneHotEncoder(handle_unknown='ignore'), CAT_LOW)]
    if te:
        pasos.append(('te', TargetEncoder(random_state=42), CAT_TE))
    else:
        pasos.append(('ohe2', OneHotEncoder(handle_unknown='ignore'), CAT_TE))
    return ColumnTransformer(pasos)

# copia con categóricas como str para CatBoost (categorías nativas)
CATS = CAT_LOW + CAT_TE
Xc = X.copy(); Xc[CATS] = Xc[CATS].astype(str)

## 4. Modelado

### 4.1 Marco de validación: OOF multi-semilla

Para cada modelo generamos su predicción **out-of-fold** (5-fold: cada fila se predice con un modelo que no
la ha visto). El RMSE de esas predicciones es una estimación honesta, y además las predicciones OOF son la
materia prima para ajustar los pesos del blend. Dentro de cada miembro, **promediamos 3 semillas** del modelo
(mismo modelo, distinta aleatoriedad interna): reduce varianza casi gratis.

In [5]:
MS = (42, 7, 777)          # semillas de modelo promediadas dentro de cada miembro

def rmse(p): return root_mean_squared_error(y, p)

def oof(fit_predict, seed_folds):
    kf = KFold(5, shuffle=True, random_state=seed_folds)
    p = np.zeros(len(X))
    for tr_idx, va_idx in kf.split(X):
        p[va_idx] = fit_predict(tr_idx, va_idx)
    return p

def make_skl(model_fn, log=False, seeds=MS):
    '''Miembro sklearn: media de `seeds` modelos; opcionalmente entrena en log1p.'''
    def fit_predict(tr_idx, va_idx):
        preds = []
        for s in seeds:
            m = model_fn(s)
            y_tr = np.log1p(y.iloc[tr_idx]) if log else y.iloc[tr_idx]
            m.fit(X.iloc[tr_idx], y_tr)
            pr = m.predict(X.iloc[va_idx])
            preds.append(np.expm1(pr) if log else pr)
        return np.mean(preds, axis=0)
    return fit_predict

def make_cat(params, log=False, seeds=MS):
    '''Miembro CatBoost con categoricas nativas.'''
    def fit_predict(tr_idx, va_idx):
        preds = []
        for s in seeds:
            m = CatBoostRegressor(**{**params, 'random_seed': s}, cat_features=CATS,
                                  verbose=0, allow_writing_files=False)
            y_tr = np.log1p(y.iloc[tr_idx]) if log else y.iloc[tr_idx]
            m.fit(Xc.iloc[tr_idx], y_tr)
            pr = m.predict(Xc.iloc[va_idx])
            preds.append(np.expm1(pr) if log else pr)
        return np.mean(preds, axis=0)
    return fit_predict

### 4.2 Los candidatos y su bake-off

Los hiperparámetros de cada candidato salen de mini-búsquedas aleatorias **validadas en folds independientes**
(el protocolo anti-*winner's curse* del notebook 03: lo que gana en la semilla de búsqueda debe confirmar en otra).
Aquí llegan ya congelados, y el bake-off los mide con OOF en **dos semillas de folds** (42 y 7) — que son también
las que usaremos para ajustar los pesos del blend.

In [6]:
XGB_P  = dict(n_estimators=900, learning_rate=0.07, max_depth=3, subsample=0.9,
              colsample_bytree=1.0, reg_lambda=1.0, min_child_weight=1, n_jobs=-1)
LGBM_P = dict(n_estimators=1200, learning_rate=0.03, num_leaves=31, min_child_samples=3,
              subsample=0.9, colsample_bytree=0.5, reg_lambda=5.0, n_jobs=-1, verbose=-1)
CAT_P  = dict(iterations=1200, learning_rate=0.05, depth=4, l2_leaf_reg=3)

xgb_fn  = lambda s: Pipeline([('p', make_pre(te=False)), ('m', XGBRegressor(**XGB_P, random_state=s))])
lgbm_fn = lambda s: Pipeline([('p', make_pre()), ('m', LGBMRegressor(**LGBM_P, random_state=s))])
ridge_fn = lambda s: Pipeline([('p', make_pre(scale=True)), ('m', Ridge(alpha=10))])

miembros = {
    'xgb_raw':   make_skl(xgb_fn,  log=False),
    'xgb_log':   make_skl(xgb_fn,  log=True),
    'lgbm':      make_skl(lgbm_fn, log=False),
    'cat_raw':   make_cat(CAT_P,   log=False),
    'cat_log':   make_cat(CAT_P,   log=True),
    'ridge_log': make_skl(ridge_fn, log=True, seeds=(42,)),
}

import time
SEEDS_PESOS = (42, 7)
OOFS = {sk: {} for sk in SEEDS_PESOS}
for sk in SEEDS_PESOS:
    for nombre, fp in miembros.items():
        t0 = time.time()
        OOFS[sk][nombre] = oof(fp, sk)
        print(f'  [folds-{sk}] {nombre:10s} listo en {time.time()-t0:5.1f}s')

print('\nOOF RMSE (€) por miembro:')
print(f'{"miembro":12s}  ' + '  '.join(f'folds-{sk}' for sk in SEEDS_PESOS))
for nombre in miembros:
    vals = '   '.join(f'{rmse(OOFS[sk][nombre]):7.2f}' for sk in SEEDS_PESOS)
    print(f'{nombre:12s}  {vals}')

  [folds-42] xgb_raw    listo en   5.5s


  [folds-42] xgb_log    listo en   5.0s


  [folds-42] lgbm       listo en  16.7s


  [folds-42] cat_raw    listo en 554.7s


  [folds-42] cat_log    listo en 562.4s
  [folds-42] ridge_log  listo en   0.1s


  [folds-7] xgb_raw    listo en   5.3s


  [folds-7] xgb_log    listo en   5.0s


  [folds-7] lgbm       listo en  15.8s


  [folds-7] cat_raw    listo en 580.8s


  [folds-7] cat_log    listo en 671.6s
  [folds-7] ridge_log  listo en   0.1s

OOF RMSE (€) por miembro:
miembro       folds-42  folds-7
xgb_raw        217.06    219.86
xgb_log        230.26    228.66
lgbm           220.60    216.08
cat_raw        221.14    222.33
cat_log        223.98    229.32
ridge_log      330.38    351.71


Referencia: el notebook 03 (features v1, un solo XGBoost) daba **~257 €**. Solo con la FE v2 los candidatos
buenos ya rondan 217–230 €. La familia de producto era la feature que faltaba.

### 4.3 Blend: pesos óptimos por NNLS

Modelos distintos se equivocan **en sitios distintos** (XGB en euros, CatBoost en log, un Ridge lineal...);
promediarlos cancela parte del error. En vez de pesos a ojo, resolvemos los pesos que minimizan el error
cuadrático de la combinación sobre las predicciones OOF — mínimos cuadrados **no negativos** (NNLS) — y los
normalizamos a suma 1.

> ⚠️ Los pesos también pueden sobreajustarse a los folds donde se calculan. Antídoto: se ajustan con las
> semillas 42+7 y se **evalúan en semillas vírgenes** (4.4) que no han participado en ninguna decisión.

In [7]:
nombres = list(miembros.keys())
A_fit = np.vstack([np.column_stack([OOFS[sk][n] for n in nombres]) for sk in SEEDS_PESOS])
y_fit = np.concatenate([y.values] * len(SEEDS_PESOS))

w_raw, _ = nnls(A_fit, y_fit)
w = w_raw / w_raw.sum()

print('Pesos del blend (NNLS, normalizados):')
for n, wi in sorted(zip(nombres, w), key=lambda t: -t[1]):
    if wi > 0.005:
        print(f'  {n:12s} {wi:.3f}')

FINAL = [n for n, wi in zip(nombres, w) if wi > 0.005]
peso_total = sum(wi for n, wi in zip(nombres, w) if wi > 0.005)
W = {n: wi / peso_total for n, wi in zip(nombres, w) if wi > 0.005}   # renormaliza tras descartar ~0

Pesos del blend (NNLS, normalizados):
  xgb_raw      0.402
  lgbm         0.337
  cat_log      0.207
  ridge_log    0.053


### 4.4 Estimación honesta: semillas de folds vírgenes

Las semillas **2024 y 555** no se han usado ni para elegir hiperparámetros ni para ajustar pesos. El RMSE del
blend ahí es nuestra mejor estimación del leaderboard (y la cifra que citamos en el README).

In [8]:
print('Validación en semillas vírgenes (blend con pesos congelados):')
honest = []
for sk in (2024, 555):
    O = {n: oof(miembros[n], sk) for n in FINAL}
    b = sum(W[n] * O[n] for n in FINAL)
    mejor_solo = min(rmse(p) for p in O.values())
    honest.append(rmse(b))
    print(f'  folds-{sk}: mejor individual {mejor_solo:7.2f} €  |  BLEND {rmse(b):7.2f} €')
print(f'\nEstimación honesta del blend: {np.mean(honest):.2f} € (media de ambas semillas)')

Validación en semillas vírgenes (blend con pesos congelados):


  folds-2024: mejor individual  222.88 €  |  BLEND  218.37 €


  folds-555: mejor individual  226.60 €  |  BLEND  217.35 €

Estimación honesta del blend: 217.86 € (media de ambas semillas)


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Reentrenamos cada miembro del blend con el **100% del train** (mismo promediado de 3 semillas). El
`TargetEncoder` se ajusta ahora con todo el train y `test.csv` solo recibirá `.transform()`.

In [9]:
def fit_full_skl(model_fn, log=False, seeds=MS):
    modelos = []
    for s in seeds:
        m = model_fn(s)
        m.fit(X, np.log1p(y) if log else y)
        modelos.append(m)
    return modelos

def fit_full_cat(params, log=False, seeds=MS):
    modelos = []
    for s in seeds:
        m = CatBoostRegressor(**{**params, 'random_seed': s}, cat_features=CATS,
                              verbose=0, allow_writing_files=False)
        m.fit(Xc, np.log1p(y) if log else y)
        modelos.append(m)
    return modelos

ENTRENADOS = {}
for n in FINAL:
    if n == 'xgb_raw':   ENTRENADOS[n] = ('skl', fit_full_skl(xgb_fn, False), False)
    elif n == 'xgb_log': ENTRENADOS[n] = ('skl', fit_full_skl(xgb_fn, True), True)
    elif n == 'lgbm':    ENTRENADOS[n] = ('skl', fit_full_skl(lgbm_fn, False), False)
    elif n == 'cat_raw': ENTRENADOS[n] = ('cat', fit_full_cat(CAT_P, False), False)
    elif n == 'cat_log': ENTRENADOS[n] = ('cat', fit_full_cat(CAT_P, True), True)
    elif n == 'ridge_log': ENTRENADOS[n] = ('skl', fit_full_skl(ridge_fn, True, seeds=(42,)), True)
print('Miembros reentrenados con todo el train:', list(ENTRENADOS.keys()))

Miembros reentrenados con todo el train: ['xgb_raw', 'lgbm', 'cat_log', 'ridge_log']


---
# PARTE 2: Predicción y submission

## 6. Carga los datos de `test.csv`

In [10]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

La misma `feature_engineering_v2` (fila a fila, sin `.fit`); los encoders de cada pipeline aplican
`.transform()` con lo aprendido en train → **sin leakage**. Las familias de producto que solo existan en test
las absorben `handle_unknown='ignore'` (OHE) y el valor por defecto del `TargetEncoder`.

In [11]:
X_pred_fe = feature_engineering_v2(X_pred)
X_test_m = X_pred_fe[ALL]
X_test_c = X_test_m.copy(); X_test_c[CATS] = X_test_c[CATS].astype(str)
X_test_m[NUM].isna().sum().sum(), X_test_m['family'].nunique()

(np.int64(0), 62)

## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [12]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

Predicción de cada miembro (media de sus 3 semillas) → combinación con los pesos del blend.

In [13]:
def predict_member(entry):
    tipo, modelos, log = entry
    Xt = X_test_c if tipo == 'cat' else X_test_m
    preds = [m.predict(Xt) for m in modelos]
    p = np.mean(preds, axis=0)
    return np.expm1(p) if log else p

preds = sum(W[n] * predict_member(ENTRENADOS[n]) for n in FINAL)
print('Rango de predicciones: %.0f - %.0f €' % (preds.min(), preds.max()))

Rango de predicciones: 230 - 4860 €


**Remate — parche de clones exactos:** 14 filas de `test.csv` son **copias idénticas** (mismas 11 columnas de
specs) de filas de `train.csv`, y dentro del train las specs duplicadas tienen el mismo precio (mediana de
dispersión: 0 €). Para esas filas, el mejor predictor no es el modelo: es el **precio conocido del clon**.
Ganancia esperada pequeña (~0,5 €), riesgo ~nulo.

In [14]:
SPECS = ['Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
         'Memory', 'Gpu', 'OpSys', 'Weight']
firma_train = df[SPECS].astype(str).agg('|'.join, axis=1)
firma_test  = X_pred[SPECS].astype(str).agg('|'.join, axis=1)
precio_clon = firma_test.map(df.groupby(firma_train)['Price_in_euros'].mean())

n_clones = precio_clon.notna().sum()
preds_final = np.where(precio_clon.notna(), precio_clon.fillna(0), preds)
print(f'Filas clonadas parcheadas: {n_clones} / {len(X_pred)}')

submission = pd.DataFrame({'laptop_ID': X_pred['laptop_ID'], 'Price_in_euros': preds_final})
submission.head()

Filas clonadas parcheadas: 14 / 391


,laptop_ID,Price_in_euros
0,209,1527.312600
1,1281,289.000000
2,1168,368.617380
3,1231,911.016025
4,1020,1079.565486


### 8.3 Chequeador

Valida la forma y, si todo está bien, guarda el CSV listo para subir.

In [15]:
def checker(df_to_submit, sample, filename=None):
    """Valida que la submission tenga la forma requerida por Kaggle y la guarda."""
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        return
    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        return
    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission.')
        return
    if filename is None:
        from datetime import datetime
        filename = f"submission_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [16]:
checker(submission, sample, filename='submission_04_blend.csv')

 ¡Todo correcto! Submission guardada como 'submission_04_blend.csv'. ¡A Kaggle!


---
## 9. Epílogo: qué pasó en el leaderboard (añadido tras la competición)

Con las etiquetas ocultas ya evaluadas, la calibración de nuestras estimaciones quedó así:

| Submission | CV honesta | Leaderboard | Peaje CV→LB |
|---|---:|---:|---:|
| 01 Baseline lineal | 355.3 € | 356.63 € | **+1.4 €** |
| 02 Random Forest | 280.3 € | 313.16 € | +32.9 € |
| 03 XGBoost (v1) | 257.5 € | 308.62 € | +51.1 € |
| **04 Blend (v2)** | **217.9 €** | **233.85 €** | **+16.0 €** |

Dos lecturas que valen más que el propio score:

1. **El peaje CV→leaderboard no es mala suerte: mide memorización.** El baseline lineal no puede memorizar → clavó su estimación. El XGBoost v1 debía parte de su ventaja a memorizar combos de specs (identidad de producto implícita) → pagó el peaje máximo. El blend v2 pagó solo +16 € porque la FE v2 le da la identidad como **feature explícita y generalizable** (`family`, `cpu_gen`, `gpu_num`): ya no necesita memorizarla. **La feature engineering v2 no solo bajó el error — convirtió memorización en señal portátil.**

2. **La validación honesta funcionó.** La estimación del blend (217.9 €, calculada en semillas de folds que no participaron en ninguna decisión) quedó a +16 € del dato real — compatible con el ruido de muestreo del subset público (~±15-25 € en ~200 filas). Del baseline al blend: **−123 € de error real (−34 %)**, y −75 € (−24 %) sobre el mejor modelo individual.